# Example: a liquid bridge between two beads

A droplet wets two nearly touching spherical beads and forms a **capillary
bridge**: the shape of minimal surface energy at fixed liquid volume, meeting
each bead at a prescribed **contact angle** $\theta$. Bridges like this hold wet
granular media together; the force they exert depends on the gap between the
beads and on $\theta$.

The beads have radius 1 and the liquid–gas surface tension is 1, so lengths are
in bead radii, energies in surface tension × radius², and forces in surface
tension × radius. The adjustable parameters:

* `theta`: the contact angle in degrees, measured inside the liquid
  (below 90°: wetting);
* `gap`: the distance between the bead surfaces (0: touching);
* `volume`: the liquid volume.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import pysurfaceevolver as pse

pv.set_jupyter_backend("static")    # "trame" in Jupyter for interactive views
pv.global_theme.window_size = [600, 450]

## The model

Only the liquid–gas surface is made of facets. The beads are **constraints**:
the two contact lines live on the spheres
$x^2 + y^2 + (z \pm c)^2 = 1$, with $c = 1 + \mathrm{gap}/2$, and slide freely on
them. The wetted part of each bead is not meshed; it enters as integrals along
the contact line, which Evolver adds to the edges on a constraint (as in the
`mound.fe` sample, there on a plane):

* **Energy.** Wetting a unit area of bead replaces solid–gas by solid–liquid
  surface, a change of $\gamma_{SL} - \gamma_{SG} = -\cos\theta$ (Young's law).
  So the energy is the liquid area plus $-\cos\theta$ times the wetted area of
  each bead.
* **Volume.** Evolver computes a body's volume from its facets as
  $\int z\, n_z\, dA$; the wetted caps close the liquid's boundary, so their part of
  that integral is added too.

For a cap around the pole of a sphere facing the gap, both are integrals over
the azimuth $\phi$ about the $z$ axis. With $u$ the cosine of the polar angle of the
contact line seen from that pole ($u = (z + c)$ on the bottom bead,
$u = (c - z)$ on the top one),

$$A = \oint (1 - u)\, d\phi, \qquad
  \int_{\text{cap}} z\, n_z\, dA = \oint \Big[\tfrac{c}{2}(1 - u^2) - \tfrac{1}{3}(1 - u^3)\Big]\, d\phi,
  \qquad d\phi = \frac{x\,dy - y\,dx}{x^2 + y^2}.$$

The bottom contact line runs counterclockwise as the boundary of the liquid
surface (normals pointing out of the liquid), the top one clockwise, hence a
sign for the top bead.

In [ ]:
def constraint(k):
    # sphere k (1: bottom, 2: top) with the contact-line energy and volume integrals
    u = "((z + CTR)/radius)" if k == 1 else "((CTR - z)/radius)"
    area = f"(radius^2*(1 - {u}))"
    cap_volume = f"(radius^2*(CTR*(1 - {u}^2)/2 - radius*(1 - {u}^3)/3))"
    sign = "" if k == 1 else "-"
    def dphi(f, name):          # f dphi as a vector integrand
        return (f"{name}1: {sign}(-y*{f})/(x^2 + y^2)\n"
                f"{name}2: {sign}(x*{f})/(x^2 + y^2)\n{name}3: 0\n")
    return (f"formula: x^2 + y^2 + (z {'+' if k == 1 else '-'} CTR)^2 = radius^2\n"
            "energy:\n" + dphi(f"(-cos(theta*pi/180)*{area})", "e")
            + "content:\n" + dphi(cap_volume, "c"))

def start_radius(gap, volume, radius=1.0):
    # radius a of a cylinder between the beads holding `volume`:
    # V(a) = 2 pi [c a^2 - 2/3 (R^3 - (R^2 - a^2)^(3/2))]
    c = radius + gap/2
    lo, hi = 0.0, 0.999*radius
    for _ in range(60):
        a = (lo + hi)/2
        v = 2*np.pi*(c*a**2 - 2/3*(radius**3 - (radius**2 - a**2)**1.5))
        lo, hi = (a, hi) if v < volume else (lo, a)
    return a

def bridge_datafile(gap, theta, volume, radius=1.0, n=8, rings=3):
    # a cylinder of liquid between the beads, its ends on the bead surfaces
    c = radius + gap/2
    a = start_radius(gap, volume, radius)
    z0 = -c + np.sqrt(radius**2 - a**2)
    phi = 2*np.pi*np.arange(n)/n
    vertices = np.vstack([np.column_stack([a*np.cos(phi), a*np.sin(phi),
                                           np.full(n, z0 - 2*z0*i/rings)])
                          for i in range(rings + 1)])
    faces = [[i*n + j, i*n + (j + 1) % n, (i + 1)*n + (j + 1) % n, (i + 1)*n + j]
             for i in range(rings) for j in range(n)]     # normals point out of the liquid
    return pse.make_datafile(
        vertices, faces,
        bodies=[pse.Body(faces=range(len(faces)), volume=volume)],
        parameters={"radius": radius, "gap": gap, "theta": theta},
        header="#define CTR (radius + gap/2)\n",
        constraints={1: constraint(1), 2: constraint(2)},
        vertex_constraints={1: range(n), 2: range(rings*n, (rings + 1)*n)})

## Relaxing

The relaxation refines the mesh three times (to about 6000 facets), with
gradient steps in between, and ends with Newton steps. Every ten steps it
equiangulates (`u`) and averages vertices (`V`): without that, facets at the
contact line get thin and collapse when the contact line moves.

Very low contact angles at larger gaps make the contact line travel far over
the beads; there the bridge is relaxed at 40° first and `theta` is then lowered
in steps of 10°, relaxing after each.

In [ ]:
def settle(ev, callback=None):
    # relax a refined surface: gradient steps with mesh upkeep, then Newton
    for _ in range(8):
        ev.command("g 10; u; V")
        if callback: callback()
    for _ in range(3):
        ev.command("g 5; u; V")
    ev.command("g 5; hessian; hessian; hessian")
    if callback: callback()

def relax_bridge(ev, levels=3, callback=None):
    ev.command("g 20")
    for _ in range(levels):
        ev.refine()
        for _ in range(8):
            ev.command("g 10; u; V")
            if callback: callback()
    settle(ev, callback)

def liquid_bridge(gap=0.2, theta=40.0, volume=0.3, callback=None):
    ev = pse.Evolver()
    ev.load_string(bridge_datafile(gap, max(theta, 40.0), volume))
    relax_bridge(ev, callback=callback)
    while ev.parameters["theta"] > theta + 1e-9:
        ev.parameters["theta"] = max(theta, ev.parameters["theta"] - 10)
        settle(ev, callback)
    return ev

Here is a bridge relaxing from its starting cylinder. In Jupyter (trame backend)
the live view redraws during the run; these docs show its first frame, and the
relaxed bridge below.

In [ ]:
def add_beads(plotter, gap, opacity=1.0):
    c = 1 + gap/2
    for z in (-c, c):
        plotter.add_mesh(pv.Sphere(radius=1, center=(0, 0, z), theta_resolution=60,
                                   phi_resolution=60), color="sandybrown", opacity=opacity)
    plotter.camera_position = [(3.2, -3.2, 1.2), (0, 0, 0), (0, 0, 1)]

gap, theta, volume = 0.2, 40.0, 0.3

ev = pse.Evolver()
ev.load_string(bridge_datafile(gap, theta, volume))
view = ev.live_view(color="lightskyblue", opacity=0.9)
add_beads(view.plotter, gap)
relax_bridge(ev, callback=view.update)

In [ ]:
def show_bridge(ev):
    plotter = pv.Plotter()
    add_beads(plotter, ev.parameters["gap"])
    plotter.add_mesh(ev.mesh().to_pyvista(), color="lightskyblue", smooth_shading=True)
    plotter.show()

show_bridge(ev)
ev

## Checking the result

The contact angle is not imposed on the mesh: it comes out of the energy
balance at the contact line. Measuring it is a check of the whole model: at each
contact-line vertex, the angle between the liquid surface's normal (out of the
liquid) and the bead's outward normal.

The pressure is the Lagrange multiplier of the volume constraint. The force
between the beads follows from the neck, the narrowest circle in the mid-plane:
there the surface is vertical by symmetry, so the force transmitted through
that plane is $F = 2\pi r_n \gamma - \pi r_n^2\, \Delta p$ (positive: the beads
attract). It must also equal the change of energy with the gap at fixed volume,
$F = dE/d\,\mathrm{gap}$.

In [ ]:
def contact_angles(ev):
    m = ev.mesh()
    c = 1 + ev.parameters["gap"]/2
    a, b, cc = (m.vertices[m.facets[:, i]] for i in range(3))
    out_of_liquid = np.where(m.facet_bodies[:, 0] == 1, 1.0, -1.0)
    normals = np.cross(b - a, cc - a) * out_of_liquid[:, None]   # area-weighted
    vertex_normals = np.zeros_like(m.vertices)
    for k in range(3):
        np.add.at(vertex_normals, m.facets[:, k], normals)
    angles = []
    for k, z in ((1, -c), (2, c)):
        on = ev.values("vertex", f"on_constraint {k}") > 0
        bead = (m.vertices[on] - [0, 0, z])                 # bead normals (radius 1)
        nl = vertex_normals[on] / np.linalg.norm(vertex_normals[on], axis=1)[:, None]
        angles.append(np.degrees(np.arccos(np.clip((nl*bead).sum(1), -1, 1))))
    return np.concatenate(angles)

def neck_radius(ev):
    # mean radius where the liquid surface crosses the mid-plane z = 0
    m = ev.mesh()
    p, q = m.vertices[m.edges[:, 0]], m.vertices[m.edges[:, 1]]
    crossing = (p[:, 2] < 0) != (q[:, 2] < 0)
    t = p[crossing, 2] / (p[crossing, 2] - q[crossing, 2])
    x = p[crossing] + t[:, None]*(q[crossing] - p[crossing])
    return np.hypot(x[:, 0], x[:, 1]).mean()

def force(ev):
    rn, p = neck_radius(ev), ev.bodies().pressure[0]
    return 2*np.pi*rn - np.pi*rn**2*p

def unstable_directions(ev):
    # number of negative eigenvalues of the Hessian (0: a stable bridge)
    out = ev.command("eigenprobe 0")
    return int(out.split("Eigencounts:")[1].split("<")[0])

angles = contact_angles(ev)
print(f"contact angle {angles.mean():.2f}° ± {angles.std():.2f}° (theta = {theta}°)")
print(f"pressure {ev.bodies().pressure[0]:.4f}, neck radius {neck_radius(ev):.4f}, "
      f"force {force(ev):.4f}, unstable directions {unstable_directions(ev)}")

h = 0.01
dE = (liquid_bridge(gap + h, theta, volume).total_energy
      - liquid_bridge(gap - h, theta, volume).total_energy) / (2*h)
print(f"dE/dgap {dE:.4f}")

The measured angle is within about a degree of `theta` (it is estimated from
flat facets, and is less accurate at very low and very high angles), and the two
forces agree to a fraction of a percent. `eigenprobe 0` counts the negative
eigenvalues of the Hessian: none, so this bridge is stable.

## Profiles for several contact angles

The more the liquid wets the beads, the further it spreads over them and the
narrower its neck. Cross-sections through the plane $y = 0$:

In [ ]:
def profile(ev):
    # (x, z) where the liquid surface crosses the half-plane y = 0, x > 0
    m = ev.mesh()
    p, q = m.vertices[m.edges[:, 0]], m.vertices[m.edges[:, 1]]
    crossing = ((p[:, 1] < 0) != (q[:, 1] < 0)) & (p[:, 0] + q[:, 0] > 0)
    t = p[crossing, 1] / (p[crossing, 1] - q[crossing, 1])
    x = p[crossing] + t[:, None]*(q[crossing] - p[crossing])
    order = np.argsort(x[:, 2])
    return x[order, 0], x[order, 2]

fig, ax = plt.subplots(figsize=(6, 4))
c = 1 + gap/2
s = np.linspace(0, 2*np.pi, 200)
for z in (-c, c):
    ax.fill(np.cos(s), z + np.sin(s), color="sandybrown", alpha=0.5, lw=0)
for th, color in zip((20, 40, 60, 80), plt.cm.viridis(np.linspace(0, 0.9, 4))):
    x, z = profile(liquid_bridge(gap, th, volume))
    ax.plot(x, z, color=color, label=f"θ = {th}°")
    ax.plot(-x, z, color=color)
ax.set_aspect("equal")
ax.set_xlim(-0.9, 0.9)
ax.set_ylim(-0.55, 0.55)
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.legend(loc="center right", fontsize="small");

## Force against gap

Wetting liquids pull the beads together, the more strongly the smaller the
contact angle. Non-wetting liquids ($\theta \gtrsim 90°$) push them apart at small
gaps, but there the symmetric bridge is **unstable**: the Hessian has a pair of
negative eigenvalues, sideways shifts in $x$ and $y$, so the drop would rather
slide out of the gap. Such points are drawn hollow. (Each point is a fresh
bridge, about half a second each.)

In [ ]:
import warnings

gaps = np.linspace(0.0, 0.6, 7)
fig, ax = plt.subplots(figsize=(6, 3.5))
for th, color in zip((20, 60, 100), ("tab:blue", "tab:green", "tab:red")):
    forces, stable = [], []
    for g in gaps:
        with warnings.catch_warnings():   # "Hessian not positive definite": see above
            warnings.simplefilter("ignore", pse.EvolverWarning)
            b = liquid_bridge(g, th, volume)
        # every Evolver is a handle to the one engine: use b before the next build
        forces.append(force(b))
        stable.append(unstable_directions(b) == 0)
    forces, stable = np.array(forces), np.array(stable)
    ax.plot(gaps, forces, "-", color=color, label=f"θ = {th}°")
    ax.plot(gaps[stable], forces[stable], "o", color=color)
    ax.plot(gaps[~stable], forces[~stable], "o", mfc="white", color=color)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("gap")
ax.set_ylabel("force (attractive > 0)")
ax.set_title(f"liquid volume {volume}")
ax.legend();

## Exporting for finite elements

For a simulation of the liquid (flow inside the bridge, say), the liquid region
needs a **closed surface** and a **volume mesh**. Evolver's mesh is only the
liquid–gas interface: the wetted parts of the beads were integrals, not facets.
So the surface is closed with a cap on each bead, built on the contact-line
vertices themselves (so that it is watertight), and tagged by region:
1 liquid–gas interface, 2 and 3 the wetted bottom and top bead.

Every `Evolver` object is a handle to the one engine, and the sweep above left
its last bridge there; build the one to export again:

In [ ]:
ev = liquid_bridge(gap, theta, volume)

def closed_surface(ev, rings=12):
    # the liquid's boundary as a closed triangle mesh with normals pointing out
    # of the liquid: the liquid-gas facets, plus a cap on each bead made of rings
    # between the contact line and the bead's pole facing the gap
    m = ev.mesh()
    c = 1 + ev.parameters["gap"]/2
    out = m.facet_bodies[:, 0] == 1
    points = [m.vertices]
    triangles = [np.where(out[:, None], m.facets, m.facets[:, ::-1])]
    tags = [np.full(len(m.facets), 1)]
    count = len(m.vertices)
    for tag, k, zc in ((2, 1, -c), (3, 2, c)):
        centre, pole = np.array([0, 0, zc]), -np.sign(zc)
        ring = np.flatnonzero(ev.values("vertex", f"on_constraint {k}") > 0)
        ring = ring[np.argsort(np.arctan2(m.vertices[ring, 1], m.vertices[ring, 0]))]
        u = m.vertices[ring] - centre                       # unit vectors on the bead
        polar = np.arccos(np.clip(pole*u[:, 2], -1, 1))     # angle from the pole
        az = np.arctan2(u[:, 1], u[:, 0])
        rows = [ring]
        for r in range(1, rings):                           # inner rings, then the pole
            a = (1 - r/rings)*polar
            points.append(centre + np.column_stack([np.sin(a)*np.cos(az), np.sin(a)*np.sin(az),
                                                    pole*np.cos(a)]))
            rows.append(count + np.arange(len(ring)))
            count += len(ring)
        points.append((centre + [0, 0, pole])[None])
        tip, count = count, count + 1
        n = len(ring)
        t = [tri for A, B in zip(rows[:-1], rows[1:]) for j in range(n)
             for tri in ([A[j], A[(j + 1) % n], B[(j + 1) % n]], [A[j], B[(j + 1) % n], B[j]])]
        t += [[rows[-1][j], rows[-1][(j + 1) % n], tip] for j in range(n)]
        t = np.array(t)
        P = np.vstack(points)                               # orient into the bead
        normal = np.cross(P[t[:, 1]] - P[t[:, 0]], P[t[:, 2]] - P[t[:, 0]])
        outward = (normal*(P[t].mean(1) - centre)).sum(1) > 0
        t[outward] = t[outward][:, ::-1]
        triangles.append(t)
        tags.append(np.full(len(t), tag))
    return np.vstack(points), np.vstack(triangles), np.concatenate(tags)

points, triangles, tags = closed_surface(ev)
a, b, cc = (points[triangles[:, i]] for i in range(3))
area = np.linalg.norm(np.cross(b - a, cc - a), axis=1)/2
print("watertight:", pse.is_watertight(triangles))
print(f"enclosed volume {np.einsum('ij,ij->i', a, np.cross(b, cc)).sum()/6:.5f} (target {volume})")
print(f"liquid-gas area {area[tags == 1].sum():.5f} (Evolver {ev.total_area:.5f}), "
      f"wetted areas {area[tags == 2].sum():.4f}, {area[tags == 3].sum():.4f}")

[Gmsh](https://gmsh.info) turns the triangulated surface into a geometry and
meshes it: `classifySurfaces` splits it into smooth patches at the sharp
contact-line edges, `createGeometry` makes them proper surfaces, and the volume
inside is meshed with tetrahedra (HXT algorithm; it avoids slivers in the thin
wedge along the contact line). The patches get physical groups: on a bead, all
points lie at distance 1 from its centre.

In [ ]:
import os, tempfile
import gmsh, meshio

def mesh_liquid(points, triangles, tags, size=0.04, folder=None):
    folder = folder or tempfile.mkdtemp()
    surface = os.path.join(folder, "bridge_surface.msh")
    meshio.write(surface, meshio.Mesh(points, [("triangle", triangles)],
                                      cell_data={"gmsh:physical": [tags],
                                                 "gmsh:geometrical": [tags]}),
                 file_format="gmsh22", binary=False)
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.merge(surface)
    gmsh.model.mesh.classifySurfaces(40*np.pi/180, True, True, np.pi)
    gmsh.model.mesh.createGeometry()
    patches = [s for _, s in gmsh.model.getEntities(2)]
    gmsh.model.geo.addVolume([gmsh.model.geo.addSurfaceLoop(patches)], 1)
    gmsh.model.geo.synchronize()
    c = 1 + gap/2
    groups = {1: [], 2: [], 3: []}
    for s in patches:
        x = gmsh.model.mesh.getNodes(2, s, includeBoundary=True)[1].reshape(-1, 3)
        on = [np.allclose(np.linalg.norm(x - [0, 0, z], axis=1), 1, atol=1e-3) for z in (-c, c)]
        groups[2 if on[0] else 3 if on[1] else 1].append(s)
    for tag, name in ((1, "interface"), (2, "bottom bead"), (3, "top bead")):
        gmsh.model.addPhysicalGroup(2, groups[tag], tag, name)
    gmsh.model.addPhysicalGroup(3, [1], 4, "liquid")
    gmsh.option.setNumber("Mesh.MeshSizeMax", size)
    gmsh.option.setNumber("Mesh.Algorithm3D", 10)
    gmsh.model.mesh.generate(3)
    gmsh.model.mesh.optimize()
    tets = gmsh.model.mesh.getElements(3)[1][0]
    quality = gmsh.model.mesh.getElementQualities(tets, "gamma")
    path = os.path.join(folder, "bridge.msh")
    gmsh.write(path)
    gmsh.finalize()
    return path, quality

path, quality = mesh_liquid(points, triangles, tags)
mesh = meshio.read(path)
tets, P = mesh.cells_dict["tetra"], mesh.points
v = np.einsum("ij,ij->i", P[tets[:, 1]] - P[tets[:, 0]],
              np.cross(P[tets[:, 2]] - P[tets[:, 0]], P[tets[:, 3]] - P[tets[:, 0]]))
print(f"{len(tets)} tetrahedra, volume {np.abs(v).sum()/6:.5f}, "
      f"quality (gamma) min {quality.min():.2f}, mean {quality.mean():.2f}")

The `.msh` file is what `dolfinx.io.gmshio.read_from_msh` reads (with the
physical groups as cell and facet tags). For other FEM codes, meshio writes the
tetrahedra and the tagged boundary triangles, for example as XDMF files:

In [ ]:
folder = os.path.dirname(path)
tag_of = mesh.cell_data_dict["gmsh:physical"]
meshio.write(os.path.join(folder, "bridge.xdmf"),
             meshio.Mesh(P, [("tetra", tets)], cell_data={"region": [tag_of["tetra"]]}))
meshio.write(os.path.join(folder, "bridge_boundary.xdmf"),
             meshio.Mesh(P, [("triangle", mesh.cells_dict["triangle"])],
                         cell_data={"boundary": [tag_of["triangle"]]}))
print(sorted(f for f in os.listdir(folder) if f.startswith("bridge")))
print("boundary triangles per tag:", dict(zip(*np.unique(tag_of["triangle"], return_counts=True))))

# a cut through the tetrahedra
grid = pv.from_meshio(meshio.Mesh(P, [("tetra", tets)]))
plotter = pv.Plotter()
plotter.add_mesh(grid.clip(normal="y", invert=False, crinkle=True), color="lightskyblue", show_edges=True)
add_beads(plotter, gap, opacity=0.3)
plotter.camera_position = [(0, -2.6, 0.4), (0, 0, 0), (0, 0, 1)]
plotter.show()

## Notes and limits

* **Changing parameters on a relaxed bridge.** `ev.parameters["theta"] = ...` or
  `["gap"]` followed by `settle(ev)` works for small changes and for lowering the
  contact angle; large jumps, or raising the angle so that the contact line has to
  retreat, can leave folded facets at the contact line (the energy is still right,
  but the measured angle is not). `liquid_bridge(...)` builds afresh each time,
  which takes about half a second.
* **Stability.** A relaxed bridge is an equilibrium, not necessarily a stable
  one: check `eigenprobe 0` (or `unstable_directions(ev)`). Non-wetting bridges
  at small gaps are saddles (see above).
* **Rupture.** At large gaps and small volumes a bridge is unstable and would
  break; the mesh cannot follow a pinch-off, so the relaxation fails there (for
  example volume 0.05 at gap 0.5).
* **Unequal beads, gravity, …** follow the same pattern: another sphere radius or
  centre in the constraint formulas, `gravity` and body densities in the datafile
  header.